# Using TruncationScheme Classes
## Objective

The objective of this tutorial is to learn how to fully utilize `truncate()` method at both QMzymeRegion and GenerateModel levels. TruncationSchemes is an abstract base class designed to assist users in constructing chemically meaningful QMzymeRegion objects. In addition, TruncationSchemes subclasses are utilized to reduce the number of atoms and electrons within the system by removing atoms while maintaining chemical accuracy. Unfortunately, when it comes to truncating the QM region, there is no "one size fits all" answer. Thus, this cookbook will try to show multiple ways in which the TruncationSchemes abstract base class can be used to truncate your model.

This workflow allows you to:

- Learn and utilize QMzymeRegion and GenerateModel level truncate() methods.

In this specific example, we are using ketosteroid isomerase (KSI) as the model system. The structure for KSI is obtained from the PDB [1OH0](https://doi.org/10.2210/pdb1OH0/pdb) and MM-minimized prior to this tutorial.

## Classes used in this example

- [GenerateModel](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.GenerateModel.html)
- [SelectionSchemes](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.SelectionSchemes.html)
    - [DistanceCutoff SelectionScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.SelectionSchemes.html#QMzyme.SelectionSchemes.DistanceCutoff)
- [CalculateModel](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.CalculateModel.html)
    - [QM_Method](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.CalculateModel.html#qm-treatment)
- [TruncationSchemes](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.TruncationScheme)
    - [TerminalAlphaCarbon TruncationScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.TerminalAlphaCarbon)
    - [AlphaCarbon TruncationScheme](https://qmzyme.readthedocs.io/en/latest/API/QMzyme.TruncationSchemes.html#QMzyme.TruncationSchemes.AlphaCarbon)

## Required Files

To start, you will need:

- A fully prepped and protonated PDB



In [ ]:
# Here are the necesary imports for this tutorial!

import QMzyme 
import pandas as pd
from QMzyme.data import PDB
from QMzyme.SelectionSchemes import DistanceCutoff

## GenerateModel.truncate()

`GenerateModel.truncate()` method can be used to truncate the current model instance. This method requires users to have methods assigned to the region prior to using `GenerateModel.truncate()`. The default truncation scheme of `GenerateModel.truncate()` is `TerminalAlphaCarbon()`. When truncation is applied to all protein residues within the CalculateModel, it will received truncated attribute.


In [17]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion test_region has an estimated charge of -2.


In [18]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[],None,None,QM,A
1,40,ASP,-1,[],[],[],None,None,QM,A
2,103,ASH,0,[],[],[],None,None,QM,A
3,263,EQU,-1,[],[],[],None,None,QM,A


All QMzymeModel regions with assigned methods will be combined and truncated according to the specified scheme. The resulting region will be saved as `CalculateModel.calc_type}_region` if name = None. The newly created region will contain information about the parent region, as well as the truncation scheme used to treat the parent region to acquire the current region.

In [19]:
model.truncate()


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [20]:
model.print_overview()

-----------------------------
Model Overview: 1oh0 
-----------------------------
  - total atoms: 4258
  - total residues: 324
  - total regions: 2
-----------------------------
Region Overview
-----------------------------
Region Name: test_region
  - atoms: 83
  - residues: 4
  - method: {'type': 'QM', 'qm_input': '6-31G* wB97XD OPT FREQ', 'basis_set': '6-31G*', 'functional': 'wB97XD', 'qm_end': '', 'program': 'gaussian', 'freeze_atoms': [], 'mult': 1, 'charge': -2}
  - selection_scheme: resid 16 40 103 263
-----------------------------
Region Name: QM_region
  - atoms: 77
  - residues: 4
  - method: {'type': 'QM', 'qm_input': '6-31G* wB97XD OPT FREQ', 'basis_set': '6-31G*', 'functional': 'wB97XD', 'qm_end': '', 'program': 'gaussian', 'freeze_atoms': [], 'mult': 1, 'charge': -2}
  - parent_region: <QMzymeRegion test_region contains 83 atom(s) and 4 residue(s)>
  - truncation_scheme: ['TerminalAlphaCarbon']
-----------------------------


When looking at the summary table, you can see that "Truncation scheme" has been updated to `TerminalAlphaCarbon` as well, for the residues that have been truncated.

In [21]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,QM,A
1,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,QM,A
2,103,ASH,0,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,QM,A
3,263,EQU,-1,[],[],[],None,None,QM,A


## QMzymeRegion.truncate()

`QMzymeRegion.truncate()` method can be used to truncate the QMzymeRegion instance. This method requires users to select a specific residue and truncation scheme to be used for the truncation. Contrasting `GenerateModel.truncate()` method, `QMzymeRegion.truncate()` method is only responsible for truncation and will not create a CalculateModel instance. In addition, if the name parameter is not specified, truncation will be applied to the self region.

In [7]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


In [8]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[],None,None,None,A
1,40,ASP,-1,[],[],[],None,None,None,A
2,103,ASH,0,[],[],[],None,None,None,A
3,263,EQU,-1,[],[],[],None,None,None,A


In [9]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.TerminalAlphaCarbon, selection="all")

After truncation, the truncation scheme attribute of the region will be updated with the one that is used for the region-level truncation.

In [12]:
model.print_overview()

-----------------------------
Model Overview: 1oh0 
-----------------------------
  - total atoms: 4258
  - total residues: 324
  - total regions: 1
-----------------------------
Region Overview
-----------------------------
Region Name: test_region
  - atoms: 77
  - residues: 4
  - method: None
  - selection_scheme: resid 16 40 103 263
  - truncation_scheme: ['TerminalAlphaCarbon']
-----------------------------


When looking at the summary table, you can see that "Truncation scheme" has been updated to `TerminalAlphaCarbon` as well, for the residues that have been truncated.

In [10]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,None,A
1,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,None,A
2,103,ASH,0,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,None,A
3,263,EQU,-1,[],[],[],None,None,None,A


## Incomplete truncation

If truncation leaves one or more protein residues untruncated, the QMzymeRegion or CalculateModel instance will not have its truncated attribute set to True. When this happens, generating an input file will raise a warning indicating that the model is only partially truncated.

In [3]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.

	Nonconventional Residues Found
	------------------------------
	EQU --> Charge: UNK, defaulting to 0

You can update charge information for nonconventional residues by running 
	>>>QMzyme.data.residue_charges.update({'3LETTER_RESNAME':INTEGER_CHARGE}). 
Note your changes will not be stored after you exit your session. It is recommended to only alter the residue_charges dictionary. If you alter the protein_residues dictionary instead that could cause unintended bugs in other modules (TruncationSchemes).

QMzymeRegion test_region has an estimated charge of -2.


For this example workflow, we will only truncate resid 16 and 40 with `TerminalAlphaCarbon` TruncationSchemes subclass. Since Ash103 has not been truncated yet, using `model.write_input()` will raise a warning statement.

In [4]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.TerminalAlphaCarbon, selection="resid 16 40")

In [5]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,QM,A
1,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[],TerminalAlphaCarbon,cap_H,QM,A
2,103,ASH,0,[],[],[],None,None,QM,A
3,263,EQU,-1,[],[],[],None,None,QM,A


In [6]:
model.write_input()


Please truncate [<QMzymeResidue resname: ASH, resid: 103, chain: X>] using GenerateModel.truncate() or QMzymeRegion.truncate().
Use of this Writer class requires citing the following: 
 	1. Alegre‐Requena, J. V., Sowndarya S. V., S., Pérez‐Soto, R., Alturaifi, T. M. & Paton, R. S. AQME: Automated quantum mechanical environments for researchers and educators. WIREs Comput Mol Sci 13, e1663 (2023).


## Ethane and methane check

When truncating a region, isolated glycine and alanine residues pose a special problem: because they have no side chain beyond a single hydrogen (Gly) or a methyl group (Ala), truncating them the same way as other residues results in the formation of methane and ethane. These small organic molecules behave differently than the side chains of glycine and alanine, which results in the final representation of the native residue not being an appropriate chemical substitute. `truncate()` checks for this during the workflow and will raise a ValueError.

In [24]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_catalytic_center(selection="resid 263")
model.set_region(selection=QMzyme.SelectionSchemes.DistanceCutoff, cutoff=3)
c_alpha_atoms = model.cutoff_3.get_atoms(attribute='name', value='CA')
model.cutoff_3.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.cutoff_3)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion cutoff_3 has an estimated charge of -2.


In [25]:
df = pd.DataFrame(model.cutoff_3.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,[],[],[CA],None,None,QM,A
1,20,VAL,0,[],[],[CA],None,None,QM,A
2,40,ASP,-1,[],[],[CA],None,None,QM,A
3,60,GLY,0,[],[],[CA],None,None,QM,A
4,61,LEU,0,[],[],[CA],None,None,QM,A
5,66,VAL,0,[],[],[CA],None,None,QM,A
6,86,PHE,0,[],[],[CA],None,None,QM,A
7,88,VAL,0,[],[],[CA],None,None,QM,A
8,90,MET,0,[],[],[CA],None,None,QM,A
9,99,LEU,0,[],[],[CA],None,None,QM,A


In this case, there is an isolated alanine residue (A118), which will result in an ethane after TerminalAlphaCarbon truncation. Thus, ValueError is raised.

In [26]:
model.truncate()

Truncation of Residue <QMzymeResidue resname: ALA, resid: 118, chain: X> would result in a(n) ethane
Representation of the native residue may not be an appropriate chemical substitute.

RESOLUTION PROTOCOL
To resolve this, either:
1) Remove them from your GenerateModel instance, run:
      GenerateModel.truncate(remove_methane=True, remove_ethane=True)
2) Set either flag to False to keep the small organic group.
3) Alternatively, you can also include the neighboring residues using
      QMzymeRegion.add_residue(resid=)
      and apply the TerminalAlphaCarbon scheme.
4) Or set extend_gly_ala_backbone=True to add ACE and NME capping to isolated Gly/Ala residue(s).


ValueError: Please set remove_methane and/or remove_ethane to True or False,
add neighboring residues, or set extend_gly_ala_backbone to True.

### Removing ethane and methane

One way to resolve this is to set remove_methane and/or remove_ethane to True. This removes the isolated glycine or alanine residues from the region entirely, rather than keeping them as a methane or ethane fragment. This is done by using the argument: `truncate(remove_methane=True, remove_ethane=True)`

In [ ]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_catalytic_center(selection="resid 263")
model.set_region(selection=QMzyme.SelectionSchemes.DistanceCutoff, cutoff=3)
c_alpha_atoms = model.cutoff_3.get_atoms(attribute='name', value='CA')
model.cutoff_3.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.cutoff_3)

In [37]:
model.truncate(remove_ethane=True)


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [38]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
1,20,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
2,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
3,60,GLY,0,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
4,61,LEU,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
5,66,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
6,86,PHE,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
7,88,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
8,90,MET,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
9,99,LEU,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A


### Keeping ethane and methane

Alternatively, you can set remove_methane and/or remove_ethane to False. This keeps the isolated glycine or alanine residues in the region as their truncated methane or ethane form, while printing a warning that these organic groups may not be an appropriate representation of the active site. This is done by using the argument: `truncate(remove_methane=False, remove_ethane=False)`.

In [39]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_catalytic_center(selection="resid 263")
model.set_region(selection=QMzyme.SelectionSchemes.DistanceCutoff, cutoff=3)
c_alpha_atoms = model.cutoff_3.get_atoms(attribute='name', value='CA')
model.cutoff_3.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.cutoff_3)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion cutoff_3 has an estimated charge of -2.


In [40]:
model.truncate(remove_ethane=True)


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [41]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
1,20,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
2,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
3,60,GLY,0,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
4,61,LEU,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
5,66,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
6,86,PHE,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
7,88,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
8,90,MET,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
9,99,LEU,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A


### Utilizing extend_gly_ala_backbone

A third option is to set extend_gly_ala_backbone=True. Instead of removing the isolated glycine or alanine residue or keeping it as a methane/ethane fragment, this caps it with acetyl (ACE) and N-methyl amide (NME) groups, extending the peptide backbone through the residue rather than truncating it down to a small organic group. This currently only works with the TerminalAlphaCarbon truncation scheme. This is done by using the argument: `truncate(extend_gly_ala_backbone=True)`.

In [42]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_catalytic_center(selection="resid 263")
model.set_region(selection=QMzyme.SelectionSchemes.DistanceCutoff, cutoff=3)
c_alpha_atoms = model.cutoff_3.get_atoms(attribute='name', value='CA')
model.cutoff_3.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.cutoff_3)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion cutoff_3 has an estimated charge of -2.


In [43]:
model.truncate(extend_gly_ala_backbone=True)


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [44]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
1,20,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
2,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
3,60,GLY,0,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
4,61,LEU,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
5,66,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
6,86,PHE,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
7,88,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
8,90,MET,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
9,99,LEU,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A


## Using multiple truncation schemes

If the user decides to truncate the QMzymeRegion with multiple truncation schemes, users can choose to apply separate truncation to two distinct groups of residues within the region or use the `override_truncation` argument to automatically truncate the non-truncated region. This section will describe how to perform both ways of truncation.

### Using separate truncate()

`QMzymeRegion.truncate()` method is capable of truncating specific residues within the QMzymeRegion instance. Using this, the user can truncate multiple sections of the region using specific truncation scheme subclasses. It is important to note that `QMzymeRegion.truncate()` method does not create CalculateModel instance, so using multi-level method need a final `GenerateModel.truncate()` to create CalculateModel instance.

In [27]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 17 40 41 103 104 263", name="test_region")


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


In [28]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.TerminalAlphaCarbon, selection="resid 16 17 103 104")

In [29]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H]",[HN],[],TerminalAlphaCarbon,cap_H,None,A
1,17,ILE,0,"[C, O]",[HC],[],TerminalAlphaCarbon,cap_H,None,A
2,40,ASP,-1,[],[],[],None,None,None,A
3,41,PRO,0,[],[],[],None,None,None,A
4,103,ASH,0,"[N, H]",[HN],[],TerminalAlphaCarbon,cap_H,None,A
5,104,VAL,0,"[C, O]",[HC],[],TerminalAlphaCarbon,cap_H,None,A
6,263,EQU,-1,[],[],[],None,None,None,A


In [30]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.AlphaCarbon, selection="resid 40 41 263")

In [31]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H]",[HN],[],TerminalAlphaCarbon,cap_H,None,A
1,17,ILE,0,"[C, O]",[HC],[],TerminalAlphaCarbon,cap_H,None,A
2,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[],AlphaCarbon,cap_H,None,A
3,41,PRO,0,"[C, O]","[HN, HC]",[],AlphaCarbon,cap_H,None,A
4,103,ASH,0,"[N, H]",[HN],[],TerminalAlphaCarbon,cap_H,None,A
5,104,VAL,0,"[C, O]",[HC],[],TerminalAlphaCarbon,cap_H,None,A
6,263,EQU,-1,[],[],[],None,None,None,A


In [32]:
model.print_overview()

-----------------------------
Model Overview: 1oh0 
-----------------------------
  - total atoms: 4258
  - total residues: 324
  - total regions: 1
-----------------------------
Region Overview
-----------------------------
Region Name: test_region
  - atoms: 126
  - residues: 7
  - method: None
  - selection_scheme: resid 16 17 40 41 103 104 263
  - truncation_scheme: ['TerminalAlphaCarbon', 'AlphaCarbon']
-----------------------------


### Using override_truncation

If you call `truncate()` method more than once and the regions involved share residues, some of those residues may already have been truncated in a previous call. override_truncation is an argument that can be used to specify how these situations are handled.

- override_truncation=None (default) raises a ValueError if the residues are already truncated, listing which residues are pre-truncated with which truncation scheme.
- override_truncation=False keeps the existing truncation on those residues and only truncates the new ones.
- override_truncation=True reverts those residues to their original, untruncated state and re-truncates them from scratch under the current call.

### override_truncation=None

If there is already a residue that is pre-truncated within the truncation selection, it will raise a ValueError with a resolving protocol.

In [17]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 17 40 41 103 104 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion test_region has an estimated charge of -2.


In [18]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.AlphaCarbon, selection="resid 16 17 103 104")

In [19]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
1,17,ILE,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
2,40,ASP,-1,[],[],[CA],None,None,QM,A
3,41,PRO,0,[],[],[CA],None,None,QM,A
4,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
5,104,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
6,263,EQU,-1,[],[],[],None,None,QM,A


In [20]:
model.truncate()

Residue <QMzymeResidue resname: TYR, resid: 16, chain: X> has already been truncated with AlphaCarbon
Residue <QMzymeResidue resname: ILE, resid: 17, chain: X> has already been truncated with AlphaCarbon
Residue <QMzymeResidue resname: ASH, resid: 103, chain: X> has already been truncated with AlphaCarbon
Residue <QMzymeResidue resname: VAL, resid: 104, chain: X> has already been truncated with AlphaCarbon
These residues will not be re-truncated by default.

RESOLUTION PROTOCOL
To override and re-truncate them, run with:
  ... .truncate(scheme=..., override_truncation=True)
or set override_truncation=False to silence this warning and skip them.


ValueError: Please set override_truncation=True to re-truncate, or False to skip.

### override_truncation=False

One way to resolve this is to set override_truncation=False. This preserves the existing truncation on residues that have already been processed and only truncates the residues that haven't been touched yet. This is done by using the argument: `truncate(override_truncation=False)`.

In [21]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 17 40 41 103 104 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion test_region has an estimated charge of -2.


In [22]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.AlphaCarbon, selection="resid 16 17 103 104")

In [23]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
1,17,ILE,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
2,40,ASP,-1,[],[],[CA],None,None,QM,A
3,41,PRO,0,[],[],[CA],None,None,QM,A
4,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
5,104,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
6,263,EQU,-1,[],[],[],None,None,QM,A


In [24]:
model.truncate(override_truncation=False)

Skipping residue <QMzymeResidue resname: TYR, resid: 16, chain: X>: it has already been truncated with AlphaCarbon.
Skipping residue <QMzymeResidue resname: ILE, resid: 17, chain: X>: it has already been truncated with AlphaCarbon.
Skipping residue <QMzymeResidue resname: ASH, resid: 103, chain: X>: it has already been truncated with AlphaCarbon.
Skipping residue <QMzymeResidue resname: VAL, resid: 104, chain: X>: it has already been truncated with AlphaCarbon.

Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [25]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
1,17,ILE,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
2,40,ASP,-1,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
3,41,PRO,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
4,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
5,104,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
6,263,EQU,-1,[],[],[],None,None,QM,A


### override_truncation=True

Another way to resolve this is to set override_truncation as True. This will revert the pre-truncated residues back to their original, untruncated state, and re-truncate them along with the residues that have not been truncated before. This is done by using the argument: `truncate(override_truncation=True)`.

In [19]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 17 40 41 103 104 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.
QMzymeRegion test_region has an estimated charge of -2.


In [20]:
model.test_region.truncate(scheme=QMzyme.TruncationSchemes.AlphaCarbon, selection="resid 16 17 103 104")

In [21]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
1,17,ILE,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
2,40,ASP,-1,[],[],[CA],None,None,QM,A
3,41,PRO,0,[],[],[CA],None,None,QM,A
4,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
5,104,VAL,0,"[N, H, C, O]","[HN, HC]",[CA],AlphaCarbon,cap_H,QM,A
6,263,EQU,-1,[],[],[],None,None,QM,A


In [22]:
model.truncate(override_truncation=True)


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [23]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
1,17,ILE,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
2,40,ASP,-1,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
3,41,PRO,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
4,103,ASH,0,"[N, H]",[HN],[CA],TerminalAlphaCarbon,cap_H,QM,A
5,104,VAL,0,"[C, O]",[HC],[CA],TerminalAlphaCarbon,cap_H,QM,A
6,263,EQU,-1,[],[],[],None,None,QM,A


## Using override_capping

If you call `QMzymeRegion.add_N_terminal_ACE` or `QMzymeRegion.add_C_terminal_NME`, these functions will cap the N and C terminus of the residue. These extended capping schemes interfere with the truncation scheme. override_capping is an argument that can be used to specify how these situations are handled.

- override_capping=None (default) raises a ValueError if the residues are already capped, listing which residues are pre-capped with which capping scheme.
- override_capping=False keeps the existing caps on those residues and only caps the new ones.
- override_capping=True removes the existing caps from those residues and re-caps them during the truncation protocol.

### override_capping=None

override_capping argument is set as None by default. If there is already a residue that is pre-capped, it will raise a ValueError with a resolving protocol.

In [5]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)
qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)

model.test_region.add_C_terminus_NME(resid = 16)
model.test_region.add_N_terminus_ACE(resid = 16)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.

	Nonconventional Residues Found
	------------------------------
	EQU --> Charge: UNK, defaulting to 0

You can update charge information for nonconventional residues by running 
	>>>QMzyme.data.residue_charges.update({'3LETTER_RESNAME':INTEGER_CHARGE}). 
Note your changes will not be stored after you exit your session. It is recommended to only alter the residue_charges dictionary. If you alter the protein_residues dictionary instead that could cause unintended bugs in other modules (TruncationSchemes).

QMzymeRegion test_region has an estimated charge of -2.


In [6]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,15,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD,...","[HH32, CH3, HH31, HH33]",[CH3],None,None,None,A
1,16,TYR,0,[],[],[CA],None,"cap_ACE, cap_NME",QM,A
2,17,NME,0,"[CA, HA, CB, HB, CG2, HG21, HG22, HG23, CG1, H...","[CH3, HH31, HH33, HH32]",[CH3],None,None,None,A
3,40,ASP,-1,[],[],[CA],None,None,QM,A
4,103,ASH,0,[],[],[CA],None,None,QM,A
5,263,EQU,-1,[],[],[],None,None,QM,A


In [7]:
model.truncate()

Residue <QMzymeResidue resname: TYR, resid: 16, chain: X> has been capped with cap_ACE, cap_NME
These residues will not be re-truncated by default.

RESOLUTION PROTOCOL
To override capping, run with:
  ... .truncate(scheme=..., override_capping=True)
or set override_capping=False to silence this warning and skip them.


ValueError: Please set override_capping=True to re-cap, or False to skip.

### override_capping=False

One way to resolve this is to set override_capping=False. This preserves the existing capping on residues that have already been processed and only truncates and caps the residues that haven't been touched yet. This is done by using the argument: `truncate(override_capping=False)`.

In [15]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)

model.test_region.add_C_terminus_NME(resid = 16)
model.test_region.add_N_terminus_ACE(resid = 16)

qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


In [16]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,15,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD,...","[HH32, CH3, HH31, HH33]",[CH3],None,None,QM,A
1,16,TYR,0,[],[],[CA],None,"cap_ACE, cap_NME",QM,A
2,17,NME,0,"[CA, HA, CB, HB, CG2, HG21, HG22, HG23, CG1, H...","[CH3, HH31, HH33, HH32]",[CH3],None,None,QM,A
3,40,ASP,-1,[],[],[CA],None,None,QM,A
4,103,ASH,0,[],[],[CA],None,None,QM,A
5,263,EQU,-1,[],[],[],None,None,QM,A


In [17]:
model.truncate(override_capping=False)

Skipping residue <QMzymeResidue resname: TYR, resid: 16, chain: X>: it has already been capped with cap_ACE, cap_NME or ACE+NME caps present.

Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [18]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,15,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD,...","[HH32, CH3, HH31, HH33]",[CH3],None,None,QM,A
1,16,TYR,0,[],[],[CA],None,"cap_ACE, cap_NME",QM,A
2,17,NME,0,"[CA, HA, CB, HB, CG2, HG21, HG22, HG23, CG1, H...","[CH3, HH31, HH33, HH32]",[CH3],None,None,QM,A
3,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
4,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
5,263,EQU,-1,[],[],[],None,None,QM,A


### override_capping=True

Another way to resolve this is to set override_capping as True. This will revert the pre-truncated residues back to their original, uncapped state, and re-cap and re-truncate them along with the residues that have not been truncated before. This is done by using the argument: `truncate(override_capping=True)`.

In [11]:
model = QMzyme.GenerateModel(PDB)
QMzyme.data.residue_charges.update({'EQU': -1})
model.set_region(selection="resid 16 40 103 263", name="test_region")
c_alpha_atoms = model.test_region.get_atoms(attribute='name', value='CA')
model.test_region.set_fixed_atoms(atoms=c_alpha_atoms)

model.test_region.add_C_terminus_NME(resid = 16)
model.test_region.add_N_terminus_ACE(resid = 16)

qm_method = QMzyme.QM_Method(
    basis_set='6-31G*',
    functional='wB97XD',
    qm_input='OPT FREQ',
    program='gaussian')
qm_method.assign_to_region(region=model.test_region)


Charge information not present. QMzyme will try to guess region charges based on residue names consistent with AMBER naming conventions (i.e., aspartate: ASP --> Charge: -1, aspartic acid: ASH --> Charge: 0.). See QMzyme.data.residue_charges for the full set.


In [12]:
df = pd.DataFrame(model.test_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,15,ACE,0,"[N, H, CA, HA, CB, HB2, HB3, CG, HG2, HG3, CD,...","[HH32, CH3, HH31, HH33]",[CH3],None,None,QM,A
1,16,TYR,0,[],[],[CA],None,"cap_ACE, cap_NME",QM,A
2,17,NME,0,"[CA, HA, CB, HB, CG2, HG21, HG22, HG23, CG1, H...","[CH3, HH31, HH33, HH32]",[CH3],None,None,QM,A
3,40,ASP,-1,[],[],[CA],None,None,QM,A
4,103,ASH,0,[],[],[CA],None,None,QM,A
5,263,EQU,-1,[],[],[],None,None,QM,A


In [13]:
model.truncate(override_capping=True)


Truncated model has been created and saved as QM_region and stored in QMzyme.CalculateModel.calculation under key QM. This model will be used to write the calculation input.


In [14]:
df = pd.DataFrame(model.QM_region.summarize())
df

,Resid,Resname,Charge,Removed atoms,Added atoms,Fixed atoms,Truncation scheme,Capping scheme,Method,Segids
0,16,TYR,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
1,40,ASP,-1,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
2,103,ASH,0,"[N, H, C, O]","[HN, HC]",[CA],TerminalAlphaCarbon,cap_H,QM,A
3,263,EQU,-1,[],[],[],None,None,QM,A
